In [27]:
import boto3
from botocore.exceptions import NoCredentialsError, PartialCredentialsError
import re

def test_aws_connection():
    try:
        # Initialize the S3 client
        s3_client = boto3.client('s3')
        
        # List S3 buckets
        response = s3_client.list_buckets()
        
        # Print the bucket names
        print("Connection Successful!")
        print("List of S3 Buckets:")
        for bucket in response['Buckets']:
            print(f"  - {bucket['Name']}")
    except NoCredentialsError:
        print("Error: AWS credentials not found. Please configure them.")
    except PartialCredentialsError:
        print("Error: Incomplete AWS credentials configuration.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

# Run the connection test
if __name__ == "__main__":
    test_aws_connection()


Connection Successful!
List of S3 Buckets:
  - hraj-datadog
  - hrajtest123
  - mygita
  - mynewrajdestbucket
  - rh-my-tf-test-bucket
  - sagemaker-studio-753523762929-a9we24a47x6
  - sagemaker-us-east-1-753523762929
  - tests3awstextract


In [30]:
#synchronous call
import boto3

# Initialize S3 and Textract clients
s3 = boto3.client('s3', region_name='us-east-1')
textract = boto3.client('textract', region_name='us-east-1')

# S3 bucket and document details
# bucket_name = "tests3awstextract"
document_name = "test.jpg"
# document_name = "Polyatomic_Ions_Quiz.pdf"

try:
    # Verify S3 object existence
    s3.head_object(Bucket=bucket_name, Key=document_name)
    print("S3 object exists and is accessible.")

    # Call Textract
    response = textract.analyze_document(
        Document={'S3Object': {'Bucket': bucket_name, 'Name': document_name}},
        FeatureTypes=['FORMS']
    )

    print("Textract operation successful!")
    for block in response['Blocks']:
        if block['BlockType'] == 'LINE':
            # print(block['Text'])
            extracted_text += block['Text'] + "\n"
            extracted_text=block['Text']
    
    child_name_match = re.search(r"Name of Child\s*:\s*(.+)", extracted_text, re.IGNORECASE)
    print({child_name_match.group(1).strip()})
except Exception as e:
    print(f"Error: {e}")

S3 object exists and is accessible.
Textract operation successful!
Error: 'NoneType' object has no attribute 'group'


In [ ]:
# asynchronous call
import boto3
import re
# Initialize S3 and Textract clients
s3 = boto3.client('s3', region_name='us-east-1')
textract = boto3.client('textract', region_name='us-east-1')

# bucket_name = "tests3awstextract"
document_name = "test.jpg"
document_name = "Polyatomic_Ions_Quiz.pdf"
try:
    # Verify S3 object existence
    s3.head_object(Bucket=bucket_name, Key=document_name)
    print("S3 object exists and is accessible.")

    # Start asynchronous Textract job for PDFs or large documents
    response = textract.start_document_analysis(
        DocumentLocation={'S3Object': {'Bucket': bucket_name, 'Name': document_name}},
        FeatureTypes=['FORMS']
    )

    job_id = response['JobId']
    print(f"Textract job started. Job ID: {job_id}")

    # Poll for job completion
    while True:
        job_status = textract.get_document_analysis(JobId=job_id)
        status = job_status['JobStatus']
        if status in ['SUCCEEDED', 'FAILED']:
            break
        print("Job in progress...")

    if status == 'SUCCEEDED':
        print("Textract operation successful!")
        for block in job_status['Blocks']:
            if block['BlockType'] == 'LINE':
                print(block['Text'])
                extracted_text=block['Text']
                re.search(r"Name of Child\s*:\s*(.+)", extracted_text, re.IGNORECASE)
    else:
        print("Textract job failed.")

except Exception as e:
    print(f"Error: {e}")


In [25]:
import boto3
import re

# Initialize S3 and Textract clients
s3 = boto3.client('s3', region_name='us-east-1')
textract = boto3.client('textract', region_name='us-east-1')

bucket_name = "tests3awstextract"
document_name = "test.jpg"

try:
    # Verify S3 object existence
    s3.head_object(Bucket=bucket_name, Key=document_name)
    print("S3 object exists and is accessible.")

    # Start asynchronous Textract job for PDFs or large documents
    response = textract.start_document_analysis(
        DocumentLocation={'S3Object': {'Bucket': bucket_name, 'Name': document_name}},
        FeatureTypes=['FORMS']
    )

    job_id = response['JobId']
    print(f"Textract job started. Job ID: {job_id}")

    # Poll for job completion
    while True:
        job_status = textract.get_document_analysis(JobId=job_id)
        status = job_status['JobStatus']
        if status in ['SUCCEEDED', 'FAILED']:
            break
        print("Job in progress...")

    if status == 'SUCCEEDED':
        print("Textract operation successful!")

        # Extract text and search for keywords
        extracted_text = ""
        for block in job_status['Blocks']:
            if block['BlockType'] == 'LINE':
                extracted_text += block['Text'] + "\n"

        # Extract "Name of Child" using regular expressions
        child_name_match = re.search(r"Name of Child\s*:\s*(.+)", extracted_text, re.IGNORECASE)

        if child_name_match:
            print(f"Extracted Name of Child: {child_name_match.group(1).strip()}")
        else:
            print("Name of Child not found.")

        if name_match:
            print(f"Extracted Name: {name_match.group(1)}")
        else:
            print("Name not found.")

        if ssn_match:
            print(f"Extracted SSN: {ssn_match.group(1)}")
        else:
            print("SSN not found.")

    else:
        print("Textract job failed.")

except Exception as e:
    print(f"Error: {e}")


S3 object exists and is accessible.
Textract job started. Job ID: 214ef068c67e1c2d6c2a30d08900ba2306b15017054dd58ed2a8fb3e543e5d35
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progress...
Job in progres

In [9]:
try:
    response = textract.list_document_classifiers()
    print("Textract connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")


Connection failed: 'Textract' object has no attribute 'list_document_classifiers'


In [16]:
import boto3

# Initialize Textract client with region
textract = boto3.client('textract', region_name='us-east-1')  # Replace 'us-east-1' with your region

try:
    # Attempt a basic Textract API call to verify connection
    response = textract.detect_document_text(
        Document={'Bytes': b'Test'}
    )
    print("Textract connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")


Connection failed: An error occurred (UnsupportedDocumentException) when calling the DetectDocumentText operation: Request has unsupported document format


In [11]:
# import boto3
with open("test.jpg", "rb") as document:
    response = textract.detect_document_text(Document={'Bytes': document.read()})


In [12]:
response

{'DocumentMetadata': {'Pages': 1},
 'Blocks': [{'BlockType': 'PAGE',
   'Geometry': {'BoundingBox': {'Width': 1.0,
     'Height': 0.9998798966407776,
     'Left': 0.0,
     'Top': 0.0001201169507112354},
    'Polygon': [{'X': 0.0, 'Y': 0.0001201169507112354},
     {'X': 1.0, 'Y': 0.0008357516489923},
     {'X': 1.0, 'Y': 1.0},
     {'X': 0.0, 'Y': 1.0}]},
   'Id': '7f9773d1-710f-464f-b402-37da9cadbfee',
   'Relationships': [{'Type': 'CHILD',
     'Ids': ['a2cffd27-37da-4763-a763-9a52a5c51b18',
      '47cff143-1bfb-44e1-ae53-d37ee129f8e3',
      'e3656520-3dc0-432f-87ac-f4742122f194',
      'd2e932f2-c1e4-4e1d-b286-2fabec97ef4e',
      '0a4fd2ae-19fe-4a6c-a10a-746ec2243d2a',
      '623cc80e-cd56-4839-aa90-a9ea602c95bc',
      'ad061a31-eba6-4bdb-85ed-cc17e24b0292',
      '375984f7-6503-4ba7-b1ca-83cf6187d441',
      '8ce75447-707e-481f-88f5-0b60cfc2c1a9',
      'a293588f-61a8-48f8-bd02-17a639571dc7',
      'b7d3d8b3-50ed-46b7-88fb-130bf6321962',
      '2731d028-fd12-4047-962e-102c626c97